# Exploração dos dados sintéticos

Análise visual do dataset gerado por `pi_server.ml.dataset` e dos modelos treinados.

**Pré-requisitos**:
```sh
uv run -m pi_server.ml.dataset
uv run -m pi_server.ml.train
```

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

sns.set_theme(style="whitegrid")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET = ROOT / "assets" / "dataset.csv"
MODELS = ROOT / "assets" / "models"

df = pd.read_csv(DATASET)
df.head()

## 1. Distribuição das features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["temperature", "humidity", "light_level"]):
    sns.histplot(df[col], bins=40, ax=ax, kde=True)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 2. Distribuição das labels

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df["ac_action"].value_counts().plot.bar(ax=axes[0], title="AC", rot=0)
df["blinds_action"].value_counts().plot.bar(ax=axes[1], title="Blinds", rot=0)
plt.tight_layout()
plt.show()

## 3. Scatter temperature × light_level coloridos pelas labels

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(
    data=df, x="temperature", y="light_level",
    hue="ac_action", alpha=0.5, ax=axes[0], s=15,
)
axes[0].set_title("Cor = AC action")
sns.scatterplot(
    data=df, x="temperature", y="light_level",
    hue="blinds_action", alpha=0.5, ax=axes[1], s=15,
)
axes[1].set_title("Cor = Blinds action")
plt.tight_layout()
plt.show()

## 4. Correlação entre features

In [ ]:
corr = df[["temperature", "humidity", "light_level"]].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlação de Pearson")
plt.show()

## 5. Avaliação dos modelos treinados

Carrega os `.joblib` e mostra confusion matrix + feature importances.

In [ ]:
from sklearn.model_selection import train_test_split

def evaluate(bundle_path, target):
    bundle = joblib.load(bundle_path)
    model, features = bundle["model"], bundle["features"]
    X = df[features].to_numpy()
    y = df[target].to_numpy()
    _, X_test, _, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, ax=axes[0])
    axes[0].set_title(f"Confusion matrix — {target}")
    importances = pd.Series(model.feature_importances_, index=features)
    importances.sort_values().plot.barh(ax=axes[1])
    axes[1].set_title("Feature importances")
    plt.tight_layout()
    plt.show()

evaluate(MODELS / "ac.joblib", "ac_action")
evaluate(MODELS / "blinds.joblib", "blinds_action")